# PointNet (Runpod Edition) — Official Repo Launcher

This notebook mirrors the old Colab workflow for the **PointNet** baseline (system info → config → dataset fetch → repo prep → training/fine-tuning/testing → log inspection → export) while running entirely on Runpod / any GPU machine using the official `Pointnet_Pointnet2_pytorch` repository.


## Assignment checklist & literature context
- **ModelNet40 requirement**: follow the official split (9,843 train / 2,468 test) and downsample to 1,024 XYZ points as mandated in the Option 5 brief.
- **Deliverables**: log accuracy (instance + class) each epoch, save checkpoints, export the bundle for Canvas, and keep confusion matrices handy for the report.
- **Literature mapping**: PointNet (Qi et al. 2017) is the lightweight baseline reproduced here; PointNet++ and DGCNN from the same literature pool should be compared against it in the report.


In [4]:
#@title 0) System info
import os
import platform
import subprocess
import sys
from datetime import datetime

print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA in torch:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU count:', torch.cuda.device_count())
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"  - cuda:{idx} -> {props.name} ({props.total_memory/1e9:.1f} GB)")
except ImportError:
    print('PyTorch not installed; install it before running training cells.')

try:
    print(subprocess.getoutput('nvidia-smi -L'))
except Exception as exc:
    print('nvidia-smi not available:', exc)

print('Working dir:', os.getcwd())
print('Timestamp:', datetime.now())


Python: 3.12.12 | packaged by conda-forge | (main, Oct 22 2025, 23:25:55) [GCC 14.3.0]
Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
PyTorch: 2.7.0+cu128
CUDA in torch: 12.8
CUDA available: True
GPU count: 1
  - cuda:0 -> NVIDIA GeForce RTX 3060 Laptop GPU (6.4 GB)
GPU 0: NVIDIA GeForce RTX 3060 Laptop GPU (UUID: GPU-df826d4a-cb40-24c7-d594-2e8bfd558eba)
Working dir: /home/rubin/uni/comp3419_A2b/runpod_pipeline
Timestamp: 2025-11-12 15:14:39.953983


In [2]:
#@title 1) Config — set your paths / hyper-parameters
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

REPO_URL = 'https://github.com/yanx27/Pointnet_Pointnet2_pytorch.git'
REPO_BRANCH = 'master'
REPO_SUBDIR = 'Pointnet_Pointnet2_pytorch'  #@param {type:"string"}
DATA_SUBDIR = 'modelnet40_normal_resampled'  #@param {type:"string"}
LOG_NAME = 'pointnet_xyz_runpod'  #@param {type:"string"}
MODEL = 'pointnet_cls'  #@param {type:"string"}
GPU = '0'  # e.g. '0' or '0,1'

NUM_POINTS = 1024
BATCH_SIZE = 32
EPOCHS = 200
LEARNING_RATE = 1e-3
DECAY_RATE = 1e-4
PROCESS_DATA = True
USE_NORMALS = False
USE_UNIFORM_SAMPLE = False

NUM_WORKERS = 12
PIN_MEMORY = True

FINE_TUNE_EPOCHS = 300
FINE_TUNE_LR = 3e-4

APPLY_RUNPOD_PATCH = True
INSTALL_REQUIREMENTS = True
AUTO_FETCH_DATASET = True
DATASET_REPO_URL = 'https://github.com/Rubindai/comp3419_A2b.git'
DATASET_BRANCH = 'main'
DATASET_SUBDIR = 'modelnet40_normal_resampled'
DATASET_REPO_FOLDER = 'dataset_repo_comp3419'

REPO_DIR = (NOTEBOOK_DIR / REPO_SUBDIR).resolve()
DATA_PATH = (NOTEBOOK_DIR / DATA_SUBDIR).resolve()
DATASET_WORKSPACE = (NOTEBOOK_DIR / DATASET_REPO_FOLDER).resolve()
LOG_DIR = (REPO_DIR / 'log' / 'classification' / LOG_NAME)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print('Notebook dir :', NOTEBOOK_DIR)
print('Repo dir     :', REPO_DIR)
print('Dataset path :', DATA_PATH)
print('Log dir      :', LOG_DIR)


Repo dir : /home/rubin/uni/comp3419_A2b/runpod_pipeline/Pointnet_Pointnet2_pytorch
Dataset  : /home/rubin/uni/comp3419_A2b/runpod_pipeline/modelnet40_normal_resampled
Log dir  : /home/rubin/uni/comp3419_A2b/runpod_pipeline/Pointnet_Pointnet2_pytorch/log/classification/pnet2_ssg_xyz_runpod


In [5]:
#@title 2) Pull dataset repo (optional, runs only if dataset missing)
import shutil
import subprocess

if AUTO_FETCH_DATASET and not DATA_PATH.exists():
    workspace = DATASET_WORKSPACE
    if (workspace / '.git').exists():
        print('Updating dataset repo at', workspace)
        subprocess.run(['git', '-C', str(workspace), 'fetch', 'origin', DATASET_BRANCH], check=True)
        subprocess.run(['git', '-C', str(workspace), 'checkout', DATASET_BRANCH], check=True)
        subprocess.run(['git', '-C', str(workspace), 'reset', '--hard', f'origin/{DATASET_BRANCH}'], check=True)
    else:
        if workspace.exists():
            print('Dataset workspace exists but is not a git repo; removing before clone:', workspace)
            shutil.rmtree(workspace)
        workspace.parent.mkdir(parents=True, exist_ok=True)
        print('Cloning dataset repo into', workspace)
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', DATASET_BRANCH,
            DATASET_REPO_URL, str(workspace)
        ], check=True)

    src = workspace / DATASET_SUBDIR
    if not src.exists():
        raise FileNotFoundError(f'Dataset sub-directory {src} not found in cloned repo')
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    print('Copying dataset from', src, '->', DATA_PATH)
    shutil.copytree(src, DATA_PATH, dirs_exist_ok=True)
else:
    if DATA_PATH.exists():
        print('Dataset already present at', DATA_PATH)
    else:
        print('AUTO_FETCH_DATASET disabled; please provide the dataset manually.')


Cloning dataset repo into dataset_repo_comp3419


Cloning into 'dataset_repo_comp3419'...
Updating files: 100% (185794/185794), done.


Copying dataset from dataset_repo_comp3419/modelnet40_normal_resampled -> /home/rubin/uni/comp3419_A2b/runpod_pipeline/modelnet40_normal_resampled


In [ ]:
#@title 3) Clone/Pull official PointNet repo + install deps
import subprocess
import sys

if not REPO_DIR.exists():
    print('Cloning', REPO_URL, 'into', REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists — fetching latest changes')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)

print('Checking out branch', REPO_BRANCH)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)

if INSTALL_REQUIREMENTS:
    req = REPO_DIR / 'requirements.txt'
    if req.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=False)
    extras = ['h5py', 'scikit-learn', 'tqdm', 'matplotlib']
    subprocess.run([sys.executable, '-m', 'pip', 'install', *extras], check=False)


In [ ]:

#@title 4) Apply runpod-centric patch (TF32 + dataloader knobs)
import subprocess
from pathlib import Path

PATCH_FILE = (NOTEBOOK_DIR / 'patches' / 'pointnet2_runpod.patch').resolve()
if APPLY_RUNPOD_PATCH:
    if not PATCH_FILE.exists():
        print('Patch file not found at', PATCH_FILE)
    else:
        print('Applying patch from', PATCH_FILE)
        result = subprocess.run(
            ['patch', '-p1', '-N', '--input', str(PATCH_FILE)],
            cwd=REPO_DIR,
            text=True,
        )
        if result.returncode == 0:
            print('Patch applied (or already present).')
        else:
            print('Patch command exited with code', result.returncode, '- check output above (it is safe if hunks were already applied).')
else:
    print('APPLY_RUNPOD_PATCH is False — skipping patch step.')


In [ ]:
#@title 5) Ensure dataset is accessible inside the repo
import shutil
from pathlib import Path

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at {DATA_PATH}. Upload/copy it or rerun the fetch cell.')

repo_data = REPO_DIR / 'data'
repo_data.mkdir(exist_ok=True)
expected = repo_data / 'modelnet40_normal_resampled'

if expected.exists() or expected.is_symlink():
    try:
        if expected.resolve() == DATA_PATH:
            print('Dataset already linked at', expected)
        else:
            if expected.is_symlink():
                expected.unlink()
            else:
                shutil.rmtree(expected)
            expected.symlink_to(DATA_PATH)
            print('Re-linked dataset to', DATA_PATH)
    except FileNotFoundError:
        print('Stale symlink detected — recreating')
        expected.unlink(missing_ok=True)
        expected.symlink_to(DATA_PATH)
else:
    expected.symlink_to(DATA_PATH)
    print('Symlinked', expected, '->', DATA_PATH)

for fn in ['modelnet40_shape_names.txt', 'modelnet40_train.txt', 'modelnet40_test.txt']:
    p = DATA_PATH / fn
    print(f"{fn:>30}:", 'OK' if p.exists() else 'MISSING')


## Pipeline recap before training
1. **Cells 0‑5** prepare the environment (system info, config, dataset fetch, official repo clone, TF32/DataLoader patch, dataset symlink) using paths relative to this notebook.
2. **Cell 6** defines helper utilities that wrap the upstream `train_classification.py` / `test_classification.py` scripts.
3. **Cells 7‑13** run PointNet baseline training, fine-tuning, evaluation, heads/tails, accuracy plots, confusion-matrix export, and artifact bundling.


In [ ]:
#@title 6) Helper functions for training/testing/log parsing
import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

RUN_ENV = os.environ.copy()


def build_base_args(log_name, *, epoch, lr):
    args = [
        sys.executable,
        'train_classification.py',
        '--model', MODEL,
        '--log_dir', log_name,
        '--num_point', str(NUM_POINTS),
        '--batch_size', str(BATCH_SIZE),
        '--epoch', str(epoch),
        '--learning_rate', str(lr),
        '--decay_rate', str(DECAY_RATE),
        '--gpu', GPU,
        '--num_workers', str(NUM_WORKERS),
    ]
    if PROCESS_DATA:
        args.append('--process_data')
    if USE_NORMALS:
        args.append('--use_normals')
    if USE_UNIFORM_SAMPLE:
        args.append('--use_uniform_sample')
    if PIN_MEMORY:
        args.append('--pin_memory')
    return args


def run_train(log_name, *, epoch, lr, extra_args=None):
    cmd = build_base_args(log_name, epoch=epoch, lr=lr)
    if extra_args:
        cmd.extend(extra_args)
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    print('
Running (train):
', printable)
    result = subprocess.run(cmd, cwd=REPO_DIR, env=RUN_ENV)
    if result.returncode != 0:
        raise RuntimeError(f'Training command failed with exit code {result.returncode}')


def run_test(log_name, *, extra_args=None):
    cmd = [
        sys.executable,
        'test_classification.py',
        '--log_dir', log_name,
        '--num_point', str(NUM_POINTS),
        '--batch_size', str(BATCH_SIZE),
        '--gpu', GPU,
        '--num_workers', str(NUM_WORKERS),
    ]
    if USE_NORMALS:
        cmd.append('--use_normals')
    if USE_UNIFORM_SAMPLE:
        cmd.append('--use_uniform_sample')
    if PIN_MEMORY:
        cmd.append('--pin_memory')
    if extra_args:
        cmd.extend(extra_args)
    printable = ' '.join(shlex.quote(str(x)) for x in cmd)
    print('
Running (test):
', printable)
    result = subprocess.run(cmd, cwd=REPO_DIR, env=RUN_ENV)
    if result.returncode != 0:
        raise RuntimeError(f'Test command failed with exit code {result.returncode}')


def log_file(model_name):
    return LOG_DIR / 'logs' / f'{model_name}.txt'


def read_log_lines(path: Path, n=20):
    if not path.exists():
        print('Missing log file:', path)
        return
    with path.open() as f:
        lines = f.readlines()
    head = ''.join(lines[:n])
    tail = ''.join(lines[-n:])
    print(f"===== {path} (first {n}) =====
{head}")
    print(f"===== {path} (last {n}) =====
{tail}")


def parse_accuracy_from_log(path: Path):
    train_acc, test_inst_acc, test_cls_acc = [], [], []
    if not path.exists():
        return train_acc, test_inst_acc, test_cls_acc
    with path.open() as f:
        for line in f:
            if 'Train Instance Accuracy:' in line:
                try:
                    train_acc.append(float(line.strip().split(':')[-1]))
                except ValueError:
                    pass
            elif 'Test Instance Accuracy:' in line and 'Class Accuracy:' in line:
                parts = line.strip().split(':')
                try:
                    inst = float(parts[1].split(',')[0])
                    cls = float(parts[2])
                    test_inst_acc.append(inst)
                    test_cls_acc.append(cls)
                except ValueError:
                    pass
    return train_acc, test_inst_acc, test_cls_acc


In [ ]:
#@title 7) Train + Test baseline (PointNet, XYZ, 1024)
run_train(LOG_NAME, epoch=EPOCHS, lr=LEARNING_RATE)
run_test(LOG_NAME)


In [ ]:
#@title 8) Fine-tune PointNet in-place (lower LR, longer schedule)
run_train(LOG_NAME, epoch=FINE_TUNE_EPOCHS, lr=FINE_TUNE_LR)
run_test(LOG_NAME)


In [ ]:
#@title 9) Evaluate checkpoint only (no extra training)
run_test(LOG_NAME)


In [ ]:
#@title 10) Heads/Tails — quick excerpts for your report
log_path = log_file(MODEL)
read_log_lines(log_path, n=20)


In [ ]:
#@title 11) Plot accuracy curves from logs
import matplotlib.pyplot as plt

log_path = log_file(MODEL)
train_acc, test_inst_acc, test_cls_acc = parse_accuracy_from_log(log_path)
if not train_acc and not test_inst_acc:
    print('No log data yet; run the training cells first.')
else:
    plt.figure(figsize=(6, 4))
    if train_acc:
        plt.plot(train_acc, label='Train Instance Acc')
    if test_inst_acc:
        plt.plot(test_inst_acc, label='Test Instance Acc')
    if test_cls_acc:
        plt.plot(test_cls_acc, label='Test Class Acc')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:

#@title 12) Confusion matrix + per-class accuracy (PNG + CSV)
import os
import sys
import importlib
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

if str(REPO_DIR) not in sys.path:
    sys.path.append(str(REPO_DIR))
from data_utils.ModelNetDataLoader import ModelNetDataLoader

EXP_DIR = REPO_DIR / 'log' / 'classification' / LOG_NAME
CKPT = EXP_DIR / 'checkpoints' / 'best_model.pth'
if not CKPT.exists():
    print('No checkpoint found at', CKPT)
else:
    class Args:
        pass
    args = Args()
    args.use_cpu = False
    args.num_category = 40
    args.num_point = NUM_POINTS
    args.use_normals = USE_NORMALS
    args.process_data = False
    args.use_uniform_sample = USE_UNIFORM_SAMPLE

    ds_root = REPO_DIR / 'data' / 'modelnet40_normal_resampled'
    if not ds_root.exists():
        ds_root = DATA_PATH

    dataset = ModelNetDataLoader(root=str(ds_root), args=args, split='test', process_data=False)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=min(NUM_WORKERS, os.cpu_count() or 1),
        pin_memory=PIN_MEMORY,
        persistent_workers=PIN_MEMORY and NUM_WORKERS > 0,
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

    model_mod = importlib.import_module(f'models.{MODEL}')
    model = model_mod.get_model(args.num_category, normal_channel=USE_NORMALS)
    state = torch.load(CKPT, map_location='cpu')
    model.load_state_dict(state['model_state_dict'])
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()
        try:
            model = torch.compile(model)
        except Exception as exc:
            print('torch.compile skipped for eval:', exc)

    all_y, all_pred = [], []
    with torch.no_grad():
        for pts, target in loader:
            pts = pts.transpose(2, 1)
            if torch.cuda.is_available():
                pts = pts.cuda()
            logits, _ = model(pts)
            pred = logits.argmax(1).cpu().numpy()
            all_pred.append(pred)
            all_y.append(target.numpy())

    y = np.concatenate(all_y)
    p = np.concatenate(all_pred)
    cm = confusion_matrix(y, p, labels=list(range(args.num_category)))
    analysis_dir = EXP_DIR / 'analysis'
    analysis_dir.mkdir(parents=True, exist_ok=True)
    np.savetxt(analysis_dir / 'confusion_matrix.csv', cm, fmt='%d', delimiter=',')
    per_class = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    np.savetxt(analysis_dir / 'per_class_accuracy.csv', per_class, fmt='%.6f', delimiter=',')

    plt.figure(figsize=(6.5, 6))
    plt.imshow(cm, cmap='Blues')
    plt.title(f'{MODEL} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(analysis_dir / 'confusion_matrix.png', dpi=180)
    plt.show()
    print('Saved confusion matrix + per-class accuracy under', analysis_dir)


In [ ]:
#@title 13) Export artifacts bundle (logs + checkpoints)
import shutil
import subprocess
import time
from pathlib import Path

EXPORT_BASE = Path('pointnet2_artifacts')
EXPORT_BASE.mkdir(exist_ok=True)
dst = EXPORT_BASE / LOG_NAME
if dst.exists():
    shutil.rmtree(dst)
shutil.copytree(LOG_DIR, dst)
note = dst / 'NOTE.txt'
note.write_text(
    f'Exported at {time.ctime()}
'
    f'Source log dir: {LOG_DIR}
'
    'Repo commit: ' + subprocess.getoutput(f"cd '{REPO_DIR}' && git rev-parse HEAD") + '
'
)
print('Exported logs/checkpoints to', dst)
for path in sorted(dst.rglob('*')):
    if path.is_file():
        print(' -', path.relative_to(dst))
